# Search FeatureSet for therapy-days

In [1]:
from __future__ import annotations

import ast
from dataclasses import dataclass
from itertools import combinations
from typing import Dict, List, Tuple, Any, Optional

import numpy as np
import pandas as pd

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import KFold, GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

import shap
import warnings
warnings.filterwarnings('ignore')

from scipy.stats import t as student_t

import matplotlib.pyplot as plt
import seaborn as sns
import itertools
from tqdm import tqdm
import os

METRIC_NAMES = ["MSE", "PCC"]
PRIMARY_METRIC = "MSE"

METRIC_HIGHER_IS_BETTER = {
    "PCC": True,
    "MSE": False,
}


def metric_higher_is_better(metric: str) -> bool:
    if metric not in METRIC_HIGHER_IS_BETTER:
        raise ValueError(f"Unknown metric direction for: {metric}")
    return METRIC_HIGHER_IS_BETTER[metric]


def best_index_by_metric(df: pd.DataFrame, col: str, metric: str):
    return df[col].idxmax() if metric_higher_is_better(metric) else df[col].idxmin()


MODEL_COLORS_ROC = ['#81989B', '#D19246', '#B5AF8B', '#7EA4B6', '#71A682', '#86BC79', '#B8DBB3', '#4A4F7E']
METRIC_COLORS = ['#6AD1A3', '#7FBDDA', '#BBC7BE','#FFD47D', '#FFA288', '#C49892']
MODEL_COLORS_BAR = ['#6AD1A3', '#7FBDDA', '#BBC7BE', '#FFD47D', '#FFA288', '#C49892', '#84ADDC']

TRAINSET_PATH='../../datasets/train_set.csv'
EXTERNAL_DATASET_PATH='../../datasets/external-1.csv'
SHAP_KERNEL_BG = 60
SHAP_KERNEL_VAL = 120
SHAP_LINEAR_BG = 200
SAVE_DIR = "../../results/treatment_duration_mse0430"
TARGET = "Therapy_duration_days"
os.makedirs(SAVE_DIR, exist_ok=True)

# prepare

In [2]:
# -----------------------------
# feature selection
# -----------------------------
def feature_selection(df, target):

    # Medication features (fixed as input, must be included for non-resistance tasks)
    polyType_cols = ['colistin_cms_daily_freq', 'polymyxin_b_daily_freq', 'colistin_sulfate_daily_freq']
    Combination_medication_cols = [
        'carbapenem_daily_dose', 'sulbactam_daily_dose', 'tigecycline_daily_dose',
        'minocycline_daily_dose', 'vancomycin_daily_dose', 'eravacycline_daily_dose',
        'aminoglycoside_daily_dose'
    ]
    medication_features = polyType_cols + Combination_medication_cols

    time_features = ['Pre_Hospital_Days', 'Pre_ICU_Days']
    base_cols = ['Age', 'Gender', 'BMI']

    comorb_cols = ['Diabetes Mellitus', 'Hypertension', 'Heart Disease', 'Stroke', 'Malignant Tumor',
        'Chronic Kidney Disease', 'Chronic Liver Disease', 'COPD', 'Comorb_other']

    df = df.copy()
    df[comorb_cols] = df[comorb_cols].fillna(0)
    df['Comorb_count'] = df[comorb_cols].sum(axis=1)
    comorb_cols = comorb_cols + ['Comorb_count']

    immuno_cols = ['Use immunosuppressive agents', 'Neutrophil Reduction', 'HIV/AIDS',
        'Post-Transplant Status', 'Chemotherapy/Radiation', 'immuno_Other']

    support_cols = ['Resp_support', 'Oxygen_concentration']

    pre_lab_cols = ['WBC', 'N_percent', 'L_count', 'PLT', 'CRP1', 'PCT1', 'D-d', 'Cr_baseline', 'eGFR1', 'RRT', 'ALT', 'AST', 'TB', 'ALB']
    dynamic_lab_cols=[]
    lab_cols = pre_lab_cols + dynamic_lab_cols

    infection_cols = ['Infection_HAP', 'Infection_VAP']
    Coinfection_cols = ['Coinfection_G_Pos', 'Coinfection_G_Neg', 'Coinfection_Fungi']
    df[Coinfection_cols] = df[Coinfection_cols].fillna(0).astype(int)
    infection_cols = infection_cols + Coinfection_cols

    # ===== RSI mapping (R/I=1, S=0) =====
    resistance_features = ['resistance_SXT', 'resistance_KAN', 'resistance_MIN', 'resistance_TGC', 'resistance_CFP-SUL', 'resistance_TOB']
    mapping = {'R': 1, 'I': 1, 'S': 0}

    for c in resistance_features:
        if c in df.columns:
            s = df[c].astype(str).str.strip().str.upper()
            df[c] = pd.to_numeric(s.map(mapping), errors="coerce")

    # Only these two groups are collapsed in SHAP ranking / group enumeration
    group_defs = {
        "Comorbidity": [c for c in comorb_cols if c in df.columns],
        "Immunosuppression": [c for c in immuno_cols if c in df.columns],
    }

    if target == 'Target_Polymyxin':
        feature_cols = base_cols + time_features + comorb_cols + immuno_cols + support_cols + pre_lab_cols
        include_med = False
    else:
        feature_cols = (
            base_cols + comorb_cols + immuno_cols + support_cols +
            lab_cols + infection_cols + resistance_features +
            polyType_cols + Combination_medication_cols
        )
        include_med = True

    feature_cols = [c for c in feature_cols if c in df.columns]

    return feature_cols, df, medication_features, group_defs, include_med

In [3]:
# -----------------------------
# Models (REGRESSION)
# -----------------------------
def get_models(random_state: int = 42):
    models = {
        "XGBoost": XGBRegressor(
            random_state=random_state,
            n_jobs=1,
        ),
        "LightGBM": LGBMRegressor(
            random_state=random_state,
            n_jobs=1,
            verbosity=-1,
        ),
        "CatBoost": CatBoostRegressor(
            loss_function="RMSE",
            random_seed=random_state,
            verbose=0,
            allow_writing_files=False,
        ),
        "RandomForest": RandomForestRegressor(
            random_state=random_state,
            n_jobs=1,
        ),
        "LinearRegression": LinearRegression(),
        "SVR": SVR(kernel="rbf"),
        "KNN": KNeighborsRegressor(),
    }
    return models


In [4]:
# -----------------------------
# Pipeline utilities
# -----------------------------
TREE_MODELS = {"XGBoost", "LightGBM", "CatBoost", "RandomForest"}
def is_tree(model_name): 
    return model_name in TREE_MODELS


def create_pipeline(model, model_name: str):
    steps = [("imputer", SimpleImputer(strategy="median"))]

    if not is_tree(model_name):
        steps.append(("scaler", StandardScaler()))

    steps.append(("clf", model))
    return Pipeline(steps)

def get_preprocess_transformer(pipe: Pipeline):
    if "clf" not in pipe.named_steps:
        raise ValueError("Pipeline must have a 'clf' step.")
    return pipe[:-1]


In [5]:
# -----------------------------
# Metrics / CV report (REGRESSION)
# -----------------------------
def evaluate_model(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    mse = float(mean_squared_error(y_true, y_pred))
    pcc = np.corrcoef(y_true, y_pred)[0, 1] if np.std(y_true) > 0 and np.std(y_pred) > 0 else np.nan
    return {"MSE": mse, "PCC": pcc}


def cv_mean_metrics(model, model_name, X, y, cv=5, random_state=42):
    """Return OOF-based regression metrics (MSE, PCC) from 5-fold CV."""
    kfold = KFold(n_splits=cv, shuffle=True, random_state=random_state)
    oof_pred = np.zeros(len(y), dtype=float)
    oof_true = y.values.astype(float).copy()

    for tr, va in kfold.split(X):
        X_tr, X_va = X.iloc[tr], X.iloc[va]
        y_tr = y.iloc[tr].values

        pipe = create_pipeline(clone(model), model_name)
        pipe.fit(X_tr, y_tr)

        oof_pred[va] = pipe.predict(X_va).astype(float)

    return evaluate_model(oof_true, oof_pred)


def cv_metrics_with_ci(
    model, model_name: str, X: pd.DataFrame, y: pd.Series,
    cv: int = 5, random_state: int = 42
) -> Dict[str, Any]:

    kfold = KFold(n_splits=cv, shuffle=True, random_state=random_state)

    metrics_per_fold = {k: [] for k in METRIC_NAMES}
    oof_pred = np.zeros(len(y), dtype=float)
    oof_true = y.values.astype(float).copy()

    for fold, (tr, va) in enumerate(kfold.split(X), start=1):
        X_tr, X_va = X.iloc[tr], X.iloc[va]
        y_tr, y_va = y.iloc[tr].values, y.iloc[va].values

        pipe = create_pipeline(clone(model), model_name)
        pipe.fit(X_tr, y_tr)

        pred = pipe.predict(X_va).astype(float)
        oof_pred[va] = pred

        m = evaluate_model(y_va, pred)
        for k in METRIC_NAMES:
            metrics_per_fold[k].append(float(m[k]))

    # t-based CI across folds
    n = cv
    dof = n - 1
    t_val = student_t.ppf(0.975, dof) if n > 1 else 0.0

    # Use OOF-based metrics for metrics_mean (instead of per-fold mean)
    oof_metrics = evaluate_model(oof_true, oof_pred)
    metrics_mean = {k: float(oof_metrics[k]) for k in METRIC_NAMES}

    # metrics_ci: keep per-fold based CI (fold-to-fold variability)
    metrics_ci = {}
    for k in METRIC_NAMES:
        vals = np.array(metrics_per_fold[k], dtype=float)
        mu = float(np.mean(vals))
        sd = float(np.std(vals, ddof=1)) if n > 1 else 0.0
        metrics_ci[k] = (mu - t_val * sd / np.sqrt(n), mu + t_val * sd / np.sqrt(n))

        
    return {
        "metrics_per_fold": metrics_per_fold,
        "metrics_mean": metrics_mean,
        "metrics_ci": metrics_ci,
        "oof_true": oof_true,
        "oof_pred": oof_pred,
    }


# SHAP

In [6]:
# -----------------------------
# CV-SHAP for all models
# -----------------------------
def _sample_indices(n: int, k: Optional[int], seed: int) -> np.ndarray:
    if k is None or k <= 0 or k >= n:
        return np.arange(n)
    rng = np.random.RandomState(seed)
    return rng.choice(n, size=k, replace=False)


def _ensure_shap_2d(shap_vals):
    if isinstance(shap_vals, list):
        shap_vals = shap_vals[1]
    shap_vals = np.asarray(shap_vals)
    if shap_vals.ndim == 3 and shap_vals.shape[2] == 2:
        shap_vals = shap_vals[:, :, 1]
    return shap_vals


def cv_shap_importance(models, X, y, cv=5, random_state=42,
    medication_features=None, group_defs=None, exclude_medication_in_ranking=True,
    use_smote=False, verbose=True, return_oof_shap=True,
    kernel_background_samples=SHAP_KERNEL_BG, kernel_val_samples=SHAP_KERNEL_VAL, linear_background_samples=SHAP_LINEAR_BG):

    assert isinstance(X, pd.DataFrame), "X must be a pandas DataFrame"
    y = pd.Series(y).astype(float)

    medication_features = medication_features or []
    group_defs = group_defs or {"Comorbidity": [], "Immunosuppression": []}

    # map feature -> group (only two groups)
    feat_to_group = {}
    for g, cols in group_defs.items():
        for c in cols:
            if c in X.columns:
                feat_to_group[c] = g

    kfold = KFold(n_splits=cv, shuffle=True, random_state=random_state)
    results: Dict[str, Any] = {}

    for model_name, model in models.items():
        if verbose:
            print(f"\n===== CV-SHAP (pipeline): {model_name} =====")

        fold_imps = []
        list_shap = []
        list_Xt = []

        for fold, (train, val) in enumerate(kfold.split(X), start=1):
            X_train, X_val = X.iloc[train], X.iloc[val]
            y_train = y.iloc[train].values

            pipe = create_pipeline(clone(model), model_name)
            pipe.fit(X_train, y_train)

            preprocess = get_preprocess_transformer(pipe)
            clf = pipe.named_steps["clf"]

            # Transform to model feature space (numeric array)
            Xt_train = preprocess.transform(X_train)
            Xt_val = preprocess.transform(X_val)

            # choose explainer by model type
            if is_tree(model_name):
                explainer = shap.TreeExplainer(clf)
                shap_vals = explainer.shap_values(Xt_val)
                shap_vals = _ensure_shap_2d(shap_vals)
                Xt_used = Xt_val

            elif model_name == "ElasticNet":
                # background in transformed space
                idx_bg = _sample_indices(Xt_train.shape[0], linear_background_samples, seed=random_state + fold)
                Xt_bg = Xt_train[idx_bg]
                explainer = shap.LinearExplainer(clf, Xt_bg, feature_perturbation="interventional")
                shap_vals = explainer.shap_values(Xt_val)
                shap_vals = _ensure_shap_2d(shap_vals)
                Xt_used = Xt_val

            else:
                # KernelExplainer for SVM/KNN (slow): subsample background and validation
                idx_bg = _sample_indices(Xt_train.shape[0], kernel_background_samples, seed=random_state + fold)
                Xt_bg = Xt_train[idx_bg]

                idx_va = _sample_indices(Xt_val.shape[0], kernel_val_samples, seed=random_state + 1000 + fold)
                Xt_va_use = Xt_val[idx_va]

                # function expects transformed features
                def f(z):
                    z = np.asarray(z)
                    if hasattr(clf, "predict"):
                        return clf.predict(z)
                    if hasattr(clf, "decision_function"):
                        s = clf.decision_function(z)
                        return 1.0 / (1.0 + np.exp(-s))
                    raise ValueError(f"{clf.__class__.__name__} lacks predict and decision_function.")

                explainer = shap.KernelExplainer(f, Xt_bg)
                shap_vals = explainer.shap_values(Xt_va_use, nsamples="auto")
                shap_vals = _ensure_shap_2d(shap_vals)
                Xt_used = Xt_va_use

            # align with original feature columns (one-hot already, preprocess doesn't change feature count)
            if shap_vals.ndim != 2 or shap_vals.shape[1] != X.shape[1]:
                raise ValueError(
                    f"{model_name} fold {fold}: SHAP shape {shap_vals.shape} != (n, {X.shape[1]}). "
                    "Make sure X is already one-hot numeric and pipeline doesn't change feature count."
                )

            if return_oof_shap:
                list_shap.append(shap_vals)
                list_Xt.append(Xt_used)
            mean_abs = np.abs(shap_vals).mean(axis=0)
            fold_imps.append({feat: float(mean_abs[i]) for i, feat in enumerate(X.columns)})

            if verbose:
                print(f"  fold {fold}: done (val_n={len(X_val)})")

        # average across folds
        avg_imp = {feat: float(np.mean([d.get(feat, 0.0) for d in fold_imps])) for feat in X.columns}
        raw_df = (
            pd.DataFrame({"feature": list(avg_imp.keys()), "mean_abs_shap": list(avg_imp.values())})
            .sort_values("mean_abs_shap", ascending=False)
            .reset_index(drop=True)
        )

        if exclude_medication_in_ranking and medication_features:
            raw_rank_df = raw_df[~raw_df["feature"].isin(medication_features)].reset_index(drop=True)
        else:
            raw_rank_df = raw_df.copy()
            
        # group aggregation (only two groups collapsed)
        group_scores = {}
        
        for _, row in raw_rank_df.iterrows():
            feat = str(row["feature"])
            val = float(row["mean_abs_shap"])
            g = feat_to_group.get(feat, feat)
            group_scores[g] = group_scores.get(g, 0.0) + val

        grouped_df = (
            pd.DataFrame({"group": list(group_scores.keys()), "mean_abs_shap": list(group_scores.values())})
            .sort_values("mean_abs_shap", ascending=False)
            .reset_index(drop=True)
        )

        results[model_name] = {
            "feature_importance": raw_df,
            "feature_importance_grouped": grouped_df,
        }
        if return_oof_shap and list_shap:
            results[model_name]["shap_oof"] = np.vstack(list_shap)
            results[model_name]["Xt_oof"] = np.vstack(list_Xt)
            results[model_name]["feature_names"] = X.columns.tolist()

    return results

# Feature-set selection

# Stage A
- per model: search top sets(top10 feature->top7 feature set)
- get 7models x 7sets

In [7]:
# -----------------------------
# 1) group_defs -> raw columns
# -----------------------------
def build_group_to_cols(feature_cols, group_defs):
    """
    feature_cols: List[str]             raw columns used in training (already one-hot / imputer + scaler)
    return: group_to_cols: Dict[str, List[str]]    eg. {"Comorbidity":[...], "Immunosuppression":[...]}
    """
    group_to_cols = {g: cols[:] for g, cols in group_defs.items()}
    grouped_cols = set(sum(group_defs.values(), []))  # all columns already assigned to some group

    for c in feature_cols:
        if c in grouped_cols:
            continue
        group_to_cols.setdefault(c, [c])  # singleton group

    return group_to_cols


def groups_to_feature_cols(groups, group_to_cols, medication_features = None, include_med = False):
    """
    groups -> expanded raw columns (dedup, keep order).
    Optionally prepend medication_features when include_med=True.
    """
    cols = []
    for g in groups:
        cols.extend(group_to_cols.get(g, [g]))

    # de-dup keep order
    cols = list(dict.fromkeys(cols))

    if include_med and medication_features:
        cols = list(dict.fromkeys(medication_features + cols))

    return cols

In [8]:
# -----------------------------
# 2) enumerate group subsets
# -----------------------------
def enumerate_group_subsets(top_groups, max_k=10):
    n = len(top_groups)
    top_groups = top_groups[:max_k]
    for k in range(1, min(max_k, n) + 1):
        for comb in combinations(top_groups, k):
            yield list(comb)

In [9]:
# -----------------------------
# 3）score subset with 5-fold CV (REGRESSION)
# -----------------------------
def cv_score_subset(model, model_name, X_sub, y, cv=5, n_jobs=1, random_state=42, metric: str = PRIMARY_METRIC):
    """Score subset by a chosen metric (default: PRIMARY_METRIC)."""
    kfold = KFold(n_splits=cv, shuffle=True, random_state=random_state)
    oof_pred = np.zeros(len(y), dtype=float)
    oof_true = np.asarray(y).ravel().astype(float)

    for tr, va in kfold.split(X_sub):
        pipe = create_pipeline(clone(model), model_name)
        pipe.fit(X_sub.iloc[tr], y.iloc[tr])
        oof_pred[va] = pipe.predict(X_sub.iloc[va]).astype(float)

    scores = evaluate_model(oof_true, oof_pred)
    if metric not in scores:
        raise ValueError(f"Unknown metric: {metric}")
    return float(scores[metric])


In [10]:
# -----------------------------
# 4)search top sets(top10 feature->top7 feature set)
# ----------------------------- 
def search_topsets_per_model(
    model,
    model_name,
    X_train,
    y_train,
    top10_groups,
    group_to_cols,
    medication_features,
    include_med,
    cv=5,
    n_jobs=1,
    top_sets=7,
    max_k=10,
    primary_metric=PRIMARY_METRIC,
):
    """
    to per model: enumerate all subsets of topk_groups, select top_sets sorted by primary metric
    return: List[(groups, score, ncols)]
    """
    results = []
    from math import comb
    n = min(len(top10_groups), max_k)
    actual_total = sum(comb(n, k) for k in range(1, n + 1)) if n > 0 else 0
    for groups in tqdm(enumerate_group_subsets(top10_groups, max_k=max_k), desc=model_name, total=actual_total):        
        cols = groups_to_feature_cols(groups, group_to_cols, medication_features, include_med)
        score = cv_score_subset(model, model_name, X_sub=X_train[cols], y=y_train, cv=cv, n_jobs=n_jobs, metric=primary_metric)
        results.append((groups, score, len(cols)))

    results.sort(
        key=lambda x: x[1],
        reverse=metric_higher_is_better(primary_metric),
    )
    return results[:top_sets]


In [11]:
# -----------------------------
# union and dedup candidates
# ----------------------------- 
def union_dedup_candidates(per_model_top_sets):
    """ merge all models' top_sets candidates, dedup by "candidates", get <= 49 candidates """
    seen = set()
    union = []
    for model_name, lst in per_model_top_sets.items():
        for candidates, score, ncols in lst:
            key = tuple(candidates)
            if key not in seen:
                seen.add(key)
                union.append({"candidates": candidates, "seed_score": score, "ncols": ncols})
    return union

In [12]:
# =========================================================
# 3) get top7 set per model
# =========================================================
def search_candidates(X_train, y_train, feature_cols, group_defs, models, shap_results, medication_features, include_med, cv=5, top_sets=7, max_k=10, verbose=True):
    # group -> raw cols
    group_to_cols = build_group_to_cols(feature_cols, group_defs)
    per_model_top_sets = {}

    for model_name, model in models.items():
        top10 = shap_results[model_name]["feature_importance_grouped"]["group"].head(10).tolist()
        per_model_top_sets[model_name] = search_topsets_per_model(
            model=model,
            model_name=model_name,
            X_train=X_train,
            y_train=y_train,
            top10_groups=top10[:max_k],
            group_to_cols=group_to_cols,
            medication_features=medication_features,
            include_med=include_med,
            cv=cv,
            top_sets=top_sets,
            max_k=max_k,
            primary_metric=PRIMARY_METRIC,
        )
        if verbose:
            print(f"model: {model_name}, top10: {top10}")
            for i in per_model_top_sets[model_name]:
                print(i)
    return per_model_top_sets, group_to_cols

# Stage B
For each candidate set, run the "full model × CV mean" calculation to obtain the final metric for the set (7-model average).

In [13]:
def evaluate_candidates(
    candidates, models, X_train, y_train, group_to_cols,
    medication_features, include_med,
    cv=5, random_state=42, threshold=0.5
):
    rows = []
    candidate_details = {}

    for i, cand in enumerate(candidates, start=1):
        set_id = f"cand_{i:02d}"
        groups = cand["candidates"]
        cols = groups_to_feature_cols(groups, group_to_cols, medication_features, include_med)
        X_sub = X_train[cols]

        per_model_metrics = {}
        for model_name, model in models.items():
            cv_ci = cv_metrics_with_ci(model, model_name, X=X_sub, y=y_train, cv=cv, random_state=random_state)
            per_model_metrics[model_name] = evaluate_model(cv_ci["oof_true"], cv_ci["oof_pred"])
        # final metric: set's mean over models
        set_metric_mean = {}
        for k in METRIC_NAMES:
            set_metric_mean[k] = float(np.mean([per_model_metrics[m][k] for m in per_model_metrics]))

        candidate_details[set_id] = {"groups": groups, "cols": cols, "per_model_metrics": per_model_metrics}

        row = {"set_id": set_id, "groups": "|".join(groups), "n_groups": len(groups), "n_cols": len(cols)}
        for k in METRIC_NAMES:
            row[f"{k}_mean_over_models"] = set_metric_mean[k]
        rows.append(row)

    set_scores_df = pd.DataFrame(rows)

    primary_col = f"{PRIMARY_METRIC}_mean_over_models"
    set_scores_df = set_scores_df.sort_values(
        primary_col,
        ascending=not metric_higher_is_better(PRIMARY_METRIC),
    ).reset_index(drop=True)
    return set_scores_df, candidate_details

# stage C
Select the optimal feature set for each metric, with metric-specific voting and direction.

In [14]:
import numpy as np

def select_final_sets(set_scores_df, candidate_details, drop_pct=0.4, min_models_kept=5, verbose=True):
    best_set_per_metric = {}

    model_names = list(next(iter(candidate_details.values()))["per_model_metrics"].keys())
    set_ids = list(candidate_details.keys())
    n_sets = len(set_ids)
    n_keep = max(1, int(n_sets * (1 - drop_pct)))

    for k in METRIC_NAMES:
        higher_is_better = metric_higher_is_better(k)

        kept_by_model = {}
        for m in model_names:
            scores = {sid: float(candidate_details[sid]["per_model_metrics"][m][k]) for sid in set_ids}
            sorted_sets = sorted(set_ids, key=lambda s: scores[s], reverse=higher_is_better)
            threshold_score = scores[sorted_sets[n_keep - 1]]
            if higher_is_better:
                kept_by_model[m] = set(sid for sid in set_ids if scores[sid] >= threshold_score)
            else:
                kept_by_model[m] = set(sid for sid in set_ids if scores[sid] <= threshold_score)

        eligible = set(set_ids)
        for m in model_names:
            eligible = eligible & kept_by_model[m]
        eligible = list(eligible)

        if not eligible:
            eligible = [sid for sid in set_ids
                       if sum(1 for m in model_names if sid in kept_by_model[m]) >= min_models_kept]
            if verbose:
                print(f"[{k}] Intersection is empty, fallback to keeping at least {min_models_kept} models, eligible_n={len(eligible)}")

        col = f"{k}_mean_over_models"
        if eligible:
            sub = set_scores_df[set_scores_df["set_id"].isin(eligible)]
            best_set_id = sub.loc[best_index_by_metric(sub, col, k), "set_id"]
            best_set_per_metric[k] = best_set_id
        else:
            best_set_id = set_scores_df.loc[best_index_by_metric(set_scores_df, col, k), "set_id"]
            best_set_per_metric[k] = best_set_id
            if verbose:
                print(f"[{k}] No qualified set, fallback to the global best mean {best_set_id}")

    return best_set_per_metric

# stage D
5-fold on the final feature-sets

In [15]:
def evaluate_final_sets(best_set_per_metric, candidate_details, models, X_train, y_train, cv=5, use_smote=False, random_state=42):
    final_deatails = {}
    metrics_rows = []

    final_set_ids = list(dict.fromkeys(best_set_per_metric.values()))
    set_id_to_metrics = {}
    for metric, set_id in best_set_per_metric.items():
        set_id_to_metrics.setdefault(set_id, []).append(metric)
    
    for set_id in final_set_ids:
        selected_by = set_id_to_metrics[set_id]
        print(f"set_id: {set_id}, selected_by: {selected_by}")

        groups = candidate_details[set_id]["groups"]
        cols = candidate_details[set_id]["cols"]
        X_sub = X_train[cols]

        final_deatails[set_id] = {"groups": groups, "cols": cols, "selected_by": selected_by, "models": {}}

        for model_name, model in models.items():
            print(model_name,": ",end="")

            cv_ci = cv_metrics_with_ci(model, model_name, X=X_sub, y=y_train, cv=cv, random_state=random_state)

            final_deatails[set_id]["models"][model_name] = {
                "cv_ci": cv_ci,
            }

            oof_metrics = evaluate_model(cv_ci["oof_true"], cv_ci["oof_pred"])
            row = {
                "set_id": set_id, "model_name": model_name, "selected_by": "|".join(selected_by),
            }
            for k in METRIC_NAMES:
                row[k] = oof_metrics[k]
                print(f"{k}: {row[k]:.4f} | ", end="")
            print()
            metrics_rows.append(row)

    final_set_metrics = pd.DataFrame(metrics_rows)
    return final_deatails, final_set_metrics

In [16]:
def pick_best_set_id(metrics_df, primary=PRIMARY_METRIC):
    if primary not in metrics_df.columns:
        raise ValueError(f"metrics_df missing column: {primary}")
    by_set = metrics_df.groupby("set_id")[primary].mean()
    return str(by_set.idxmax() if metric_higher_is_better(primary) else by_set.idxmin())

# main

### prepare data and models

In [17]:
# ===== 1. load data =====
train_df = pd.read_csv(TRAINSET_PATH, encoding='gbk')
test_df = pd.read_csv(EXTERNAL_DATASET_PATH, encoding='gbk')

train_df = train_df[(train_df["Allcause_inhospital_death"] == 2) & (train_df["Clinical_outcome"]==1)].copy()
test_df = test_df[(test_df["Allcause_inhospital_death"] == 2) & (test_df["Clinical_outcome"]==1)].copy()

target = TARGET

train_df = train_df[train_df[target].notna()].copy()
test_df = test_df[test_df[target].notna()].copy()

print(f"✅ Using label column: {target}")
print(f"data shape: {train_df.shape}")
print(train_df[target].describe())
print(f"data shape: {test_df.shape}")
print(test_df[target].describe())

# ===== 2. feature selection =====
feature_cols, train_df, medication_features, group_defs, include_med = feature_selection(train_df, target)
X_train,y_train = train_df[feature_cols], train_df[target]

_, test_df, _, _, _ = feature_selection(test_df, target)
X_test,y_test = test_df[feature_cols], test_df[target]

# ===== 3. models =====
models = get_models(random_state=42)

✅ Using label column: Therapy_duration_days
data shape: (277, 126)
count    277.000000
mean      12.624549
std        8.834417
min        3.000000
25%        7.000000
50%       11.000000
75%       15.000000
max       57.000000
Name: Therapy_duration_days, dtype: float64
data shape: (80, 126)
count    80.000000
mean     12.762500
std       8.509479
min       2.000000
25%       7.000000
50%      11.000000
75%      16.000000
max      58.000000
Name: Therapy_duration_days, dtype: float64


# 5-fold on Trainset with all-feature

In [18]:
for model_name, model in models.items():
    m = cv_mean_metrics(model, model_name, X=X_train, y=y_train, cv=5, random_state=42)
    m_clean = {k: float(v) for k, v in m.items()}
    print(model_name, "Train (5-fold OOF):", m_clean)

XGBoost Train (5-fold OOF): {'MSE': 86.1622095880464, 'PCC': 0.14676129242160035}
LightGBM Train (5-fold OOF): {'MSE': 94.3176927215787, 'PCC': 0.038653900616983074}
CatBoost Train (5-fold OOF): {'MSE': 79.41818820268762, 'PCC': 0.11392403591296298}
RandomForest Train (5-fold OOF): {'MSE': 76.09443537906138, 'PCC': 0.20356536031872235}
LinearRegression Train (5-fold OOF): {'MSE': 109.65172549774358, 'PCC': 0.1780931560432579}
SVR Train (5-fold OOF): {'MSE': 81.28022685707981, 'PCC': 0.0659015406671977}
KNN Train (5-fold OOF): {'MSE': 79.84303249097474, 'PCC': 0.15784026477329777}


### SHAP -> feature importance rank -> top10_groups per model

In [19]:
# ===== 4. SHAP ranking (pipeline version) =====
shap_results = cv_shap_importance(
        models=models, X=X_train, y=y_train, cv=5, medication_features=medication_features,
        group_defs=group_defs, exclude_medication_in_ranking=True,return_oof_shap=True,
        verbose=False
    )

  0%|          | 0/56 [00:00<?, ?it/s]

  0%|          | 0/56 [00:00<?, ?it/s]

  0%|          | 0/55 [00:00<?, ?it/s]

  0%|          | 0/55 [00:00<?, ?it/s]

  0%|          | 0/55 [00:00<?, ?it/s]

...

  0%|          | 0/56 [00:00<?, ?it/s]

  0%|          | 0/56 [00:00<?, ?it/s]

  0%|          | 0/55 [00:00<?, ?it/s]

  0%|          | 0/55 [00:00<?, ?it/s]

  0%|          | 0/55 [00:00<?, ?it/s]

In [20]:
for model_name in models:
    top10 = shap_results[model_name]["feature_importance_grouped"]["group"].head(10).tolist()
    print(f"{model_name}: {top10}")

XGBoost: ['Comorbidity', 'PLT', 'Cr_baseline', 'N_percent', 'AST', 'L_count', 'ALB', 'D-d', 'CRP1', 'Age']
LightGBM: ['PLT', 'Comorbidity', 'ALB', 'Cr_baseline', 'D-d', 'eGFR1', 'L_count', 'Resp_support', 'BMI', 'N_percent']
CatBoost: ['PLT', 'Comorbidity', 'L_count', 'Cr_baseline', 'eGFR1', 'ALB', 'Resp_support', 'N_percent', 'BMI', 'ALT']
RandomForest: ['N_percent', 'Comorbidity', 'L_count', 'PLT', 'Cr_baseline', 'ALB', 'AST', 'D-d', 'eGFR1', 'WBC']
LinearRegression: ['Resp_support', 'Comorbidity', 'eGFR1', 'L_count', 'Cr_baseline', 'PLT', 'Coinfection_Fungi', 'resistance_MIN', 'Immunosuppression', 'ALB']
SVR: ['Comorbidity', 'resistance_MIN', 'Coinfection_G_Neg', 'Immunosuppression', 'Resp_support', 'Coinfection_Fungi', 'resistance_KAN', 'Age', 'Gender', 'resistance_TGC']
KNN: ['Comorbidity', 'Immunosuppression', 'resistance_CFP-SUL', 'resistance_MIN', 'Coinfection_Fungi', 'Coinfection_G_Neg', 'Infection_VAP', 'Infection_HAP', 'Oxygen_concentration', 'resistance_KAN']


In [21]:
shap_dir=SAVE_DIR+'/shap_csv'
os.makedirs(shap_dir, exist_ok=True)

# 1. Medication Included
all_df = pd.concat([res["feature_importance"].assign(model=mn) for mn, res in shap_results.items()], ignore_index=True)
all_df.to_csv(shap_dir + "/importance_with_medication_no_group-nogridsearch.csv", index=False)

# 2. Medication Excluded
all_df = pd.concat([
        res["feature_importance"][~res["feature_importance"]["feature"].isin(medication_features)].assign(model=mn)
        for mn, res in shap_results.items()
    ], ignore_index=True)
all_df.to_csv(shap_dir + "/importance_no_medication_no_group-nogridsearch.csv", index=False)

# 3. Grouped
all_df = pd.concat([res["feature_importance_grouped"].assign(model=mn) for mn, res in shap_results.items()], ignore_index=True)
all_df.to_csv(shap_dir + "/importance_no_medication_with_group-nogridsearch.csv", index=False)

### Get top7 feature-sets per model

In [22]:
per_model_top_sets, group_to_cols = search_candidates(
    X_train, y_train, feature_cols, group_defs, models,
    shap_results, medication_features, include_med, 
    cv=5, top_sets=7, max_k=10, verbose=True
)

XGBoost: 100%|██████████| 1023/1023 [07:15<00:00,  2.35it/s]


model: XGBoost, top10: ['Comorbidity', 'PLT', 'Cr_baseline', 'N_percent', 'AST', 'L_count', 'ALB', 'D-d', 'CRP1', 'Age']
(['PLT', 'N_percent', 'AST', 'L_count', 'ALB'], 82.71955566222199, 15)
(['Comorbidity', 'PLT', 'N_percent', 'AST', 'L_count', 'CRP1'], 83.70139539419156, 25)
(['L_count', 'ALB'], 83.93162720895164, 12)
(['PLT', 'N_percent', 'AST', 'L_count'], 84.56077360006351, 14)
(['PLT', 'Cr_baseline', 'AST', 'L_count', 'D-d'], 85.0247762733672, 15)
(['Comorbidity', 'PLT', 'N_percent', 'AST', 'L_count', 'Age'], 85.15699991737468, 25)
(['Comorbidity', 'Cr_baseline', 'N_percent', 'AST', 'L_count', 'ALB', 'D-d', 'CRP1'], 85.38584647005825, 27)


LightGBM: 100%|██████████| 1023/1023 [03:01<00:00,  5.62it/s]


model: LightGBM, top10: ['PLT', 'Comorbidity', 'ALB', 'Cr_baseline', 'D-d', 'eGFR1', 'L_count', 'Resp_support', 'BMI', 'N_percent']
(['Resp_support'], 81.19057281522127, 11)
(['ALB', 'L_count', 'Resp_support', 'BMI'], 84.42877817705086, 14)
(['Comorbidity', 'ALB', 'Resp_support'], 84.59909306740563, 22)
(['Comorbidity', 'ALB', 'D-d', 'Resp_support'], 85.04236179155664, 23)
(['Comorbidity', 'D-d', 'Resp_support'], 85.3873406100828, 22)
(['ALB', 'Resp_support', 'BMI'], 85.48555236922783, 13)
(['Comorbidity', 'ALB', 'Resp_support', 'BMI'], 85.69277296629737, 23)


CatBoost: 100%|██████████| 1023/1023 [2:19:08<00:00,  8.16s/it] 


...

model: LinearRegression, top10: ['Resp_support', 'Comorbidity', 'eGFR1', 'L_count', 'Cr_baseline', 'PLT', 'Coinfection_Fungi', 'resistance_MIN', 'Immunosuppression', 'ALB']
(['Resp_support', 'eGFR1', 'L_count', 'PLT'], 68.41351706410114, 14)
(['Resp_support', 'L_count', 'PLT'], 68.68302034571076, 13)
(['Resp_support', 'eGFR1', 'L_count', 'Cr_baseline', 'PLT'], 68.97519327780284, 15)
(['Resp_support', 'eGFR1', 'L_count', 'PLT', 'resistance_MIN'], 69.13898335400107, 15)
(['Resp_support', 'L_count', 'Cr_baseline', 'PLT'], 69.29282993606472, 14)
(['Resp_support', 'L_count', 'PLT', 'resistance_MIN'], 69.36362343014649, 14)
(['Resp_support', 'eGFR1', 'L_count', 'Cr_baseline', 'PLT', 'resistance_MIN'], 69.6920061933558, 16)


SVR: 100%|██████████| 1023/1023 [01:12<00:00, 14.12it/s]


model: SVR, top10: ['Comorbidity', 'resistance_MIN', 'Coinfection_G_Neg', 'Immunosuppression', 'Resp_support', 'Coinfection_Fungi', 'resistance_KAN', 'Age', 'Gender', 'resistance_TGC']
(['Coinfection_G_Neg', 'Immunosuppression'], 80.03896101849779, 16)
(['Coinfection_G_Neg', 'Immunosuppression', 'Gender'], 80.10246197652754, 17)
(['resistance_MIN', 'Coinfection_G_Neg', 'Immunosuppression'], 80.10990487340595, 17)
(['resistance_MIN', 'Immunosuppression', 'Gender'], 80.1131374055243, 17)
(['resistance_MIN', 'Coinfection_G_Neg', 'resistance_KAN'], 80.19669716681207, 13)
(['Coinfection_G_Neg', 'Immunosuppression', 'Resp_support', 'Gender'], 80.21604004097517, 18)
(['resistance_MIN', 'Coinfection_G_Neg', 'Immunosuppression', 'Gender'], 80.25210359593117, 18)


KNN: 100%|██████████| 1023/1023 [01:13<00:00, 13.90it/s]

model: KNN, top10: ['Comorbidity', 'Immunosuppression', 'resistance_CFP-SUL', 'resistance_MIN', 'Coinfection_Fungi', 'Coinfection_G_Neg', 'Infection_VAP', 'Infection_HAP', 'Oxygen_concentration', 'resistance_KAN']
(['Immunosuppression', 'resistance_CFP-SUL', 'resistance_MIN', 'Coinfection_Fungi', 'Infection_VAP', 'resistance_KAN'], 76.26599277978339, 20)
(['Immunosuppression', 'resistance_CFP-SUL', 'resistance_MIN', 'Coinfection_Fungi', 'Infection_HAP', 'resistance_KAN'], 76.28259927797833, 20)
(['resistance_CFP-SUL', 'Coinfection_Fungi', 'Oxygen_concentration', 'resistance_KAN'], 76.40014440433212, 14)
(['Immunosuppression', 'resistance_CFP-SUL', 'resistance_MIN', 'Infection_VAP', 'Oxygen_concentration'], 76.47090252707582, 19)
(['Immunosuppression', 'resistance_CFP-SUL', 'resistance_MIN', 'Infection_HAP', 'Oxygen_concentration'], 76.4834657039711, 19)
(['resistance_MIN', 'Infection_VAP', 'Oxygen_concentration', 'resistance_KAN'], 76.55335740072202, 14)
(['resistance_MIN', 'Infection_

### Evaluate candidates(7*7)

In [23]:
candidates = union_dedup_candidates(per_model_top_sets)
set_scores_df, candidate_details = evaluate_candidates(candidates, models, X_train, y_train, group_to_cols, medication_features, include_med, cv=5)

In [24]:
feature_selection_dir = SAVE_DIR +'/feature_selection'
os.makedirs(feature_selection_dir, exist_ok=True)
set_scores_df.to_csv(feature_selection_dir + "/set_scores_stageB-nogridsearch.csv", index=False, encoding="utf-8-sig")

In [25]:
for i in candidate_details:
    print(i, candidate_details[i])
rows = []
for set_id, d in candidate_details.items():
    groups_str = "|".join(d["groups"])
    for model_name, m in d["per_model_metrics"].items():
        row = {"set_id": set_id, "groups": groups_str, "model": model_name, **m}
        rows.append(row)
pd.DataFrame(rows).to_csv(feature_selection_dir + "/candidate_details-nogridsearch.csv", index=False, encoding="utf-8-sig")

cand_01 {'groups': ['PLT', 'N_percent', 'AST', 'L_count', 'ALB'], 'cols': ['colistin_cms_daily_freq', 'polymyxin_b_daily_freq', 'colistin_sulfate_daily_freq', 'carbapenem_daily_dose', 'sulbactam_daily_dose', 'tigecycline_daily_dose', 'minocycline_daily_dose', 'vancomycin_daily_dose', 'eravacycline_daily_dose', 'aminoglycoside_daily_dose', 'PLT', 'N_percent', 'AST', 'L_count', 'ALB'], 'per_model_metrics': {'XGBoost': {'MSE': 82.71955566222199, 'PCC': np.float64(0.26770150657381997)}, 'LightGBM': {'MSE': 94.10879472299963, 'PCC': np.float64(0.04698337015257641)}, 'CatBoost': {'MSE': 76.00787349145045, 'PCC': np.float64(0.23194212325206262)}, 'RandomForest': {'MSE': 79.07951227436821, 'PCC': np.float64(0.20707284591398128)}, 'LinearRegression': {'MSE': 78.47963679530602, 'PCC': np.float64(0.24493014663494997)}, 'SVR': {'MSE': 78.98221017490557, 'PCC': np.float64(0.19409854856317912)}, 'KNN': {'MSE': 78.12014440433212, 'PCC': np.float64(0.19589598470997707)}}}
cand_02 {'groups': ['Comorbid

### Select final feature-set by metric

In [26]:
best_set_per_metric = select_final_sets(set_scores_df, candidate_details, drop_pct=0.4, min_models_kept=6, verbose=True)
print("best_set_per_metric:", best_set_per_metric)
final_set_ids = list(dict.fromkeys(best_set_per_metric.values()))
print("final_set_ids:", final_set_ids)

best_set_per_metric: {'MSE': 'cand_16', 'PCC': 'cand_17'}
final_set_ids: ['cand_16', 'cand_17']


In [27]:
for set_id in final_set_ids:
    cols = candidate_details[set_id]["cols"]
    groups = candidate_details[set_id]["groups"]
    selected_by = [m for m, s in best_set_per_metric.items() if s == set_id]
    print(f"set_id={set_id}, selected_by={selected_by}, n_cols={len(cols)}")
    print("  cols:", cols)

set_id=cand_16, selected_by=['MSE'], n_cols=15
  cols: ['colistin_cms_daily_freq', 'polymyxin_b_daily_freq', 'colistin_sulfate_daily_freq', 'carbapenem_daily_dose', 'sulbactam_daily_dose', 'tigecycline_daily_dose', 'minocycline_daily_dose', 'vancomycin_daily_dose', 'eravacycline_daily_dose', 'aminoglycoside_daily_dose', 'PLT', 'L_count', 'Resp_support', 'BMI', 'ALT']
set_id=cand_17, selected_by=['PCC'], n_cols=15
  cols: ['colistin_cms_daily_freq', 'polymyxin_b_daily_freq', 'colistin_sulfate_daily_freq', 'carbapenem_daily_dose', 'sulbactam_daily_dose', 'tigecycline_daily_dose', 'minocycline_daily_dose', 'vancomycin_daily_dose', 'eravacycline_daily_dose', 'aminoglycoside_daily_dose', 'PLT', 'L_count', 'eGFR1', 'Resp_support', 'ALT']


### 5-fold on final feature-set

In [28]:
final_set_eval, final_set_metrics = evaluate_final_sets(best_set_per_metric, candidate_details, models, X_train, y_train, cv=5, use_smote=False)
final_set_metrics.to_csv(SAVE_DIR + "/final_set_metrics-nogridsearch.csv", index=False)

set_id: cand_16, selected_by: ['MSE']
XGBoost : MSE: 77.5766 | PCC: 0.2761 | 
LightGBM : MSE: 89.3101 | PCC: 0.1052 | 
CatBoost : MSE: 69.1800 | PCC: 0.3434 | 
RandomForest : MSE: 69.3918 | PCC: 0.3354 | 
LinearRegression : MSE: 69.6219 | PCC: 0.3496 | 
SVR : MSE: 79.4949 | PCC: 0.1783 | 
KNN : MSE: 78.0377 | PCC: 0.1946 | 
set_id: cand_17, selected_by: ['PCC']
XGBoost : MSE: 77.4704 | PCC: 0.3081 | 
LightGBM : MSE: 90.3060 | PCC: 0.1331 | 
CatBoost : MSE: 69.9899 | PCC: 0.3317 | 
RandomForest : MSE: 69.1850 | PCC: 0.3423 | 
LinearRegression : MSE: 68.8424 | PCC: 0.3613 | 
SVR : MSE: 79.3969 | PCC: 0.1733 | 
KNN : MSE: 77.8721 | PCC: 0.2015 | 
